# Human Activity Classification Under Domain and Temporal Shift

## POLAR, V-COCO, and Okutama-Action evidence

This notebook is the compact, executable evidence narrative for the repository's
still-image, person-level transfer, and short-video studies. It reads only tracked,
path-sanitized evidence and performs no training or post-evaluation selection.

**Locked result:** 0.940 macro-F1 (95% CI
[0.931, 0.948]) and
0.946 accuracy on 3,329 held-out images.

The person-level V-COCO follow-up reaches
**0.8663 official-test macro-F1**, improving over
the historical source-only DINO baseline by
**+0.1592**, with a 95% image-cluster interval of
[+0.1454, +0.1735].

On sealed Okutama confirmation data, the static target-trained model reaches
**0.7458 macro-F1** and the temporal teacher
reaches **0.7854**. Routing half of the samples
to clips retains **0.7817**.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "results" / "polar_test_summary.json").is_file():
    raise RuntimeError("Run this notebook from the repository root.")

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

def load_json(name):
    return json.loads((ROOT / "results" / name).read_text(encoding="utf-8"))

## 1. Evidence boundary

The audit removes confirmed cross-split source relatives before supervised fitting. The
selection lock then fixes every model, seed, epoch count, classifier setting, blend
weight, metric, and bootstrap rule before test access.

In [2]:
audit = load_json("polar_data_audit.json")
gate = load_json("polar_test_access_gate.json")
fits = load_json("polar_final_fit_manifest.json")

controls = pd.DataFrame(
    [
        ("Clean development images", fits["development_rows"]),
        ("Clean held-out test images", sum(audit["clean_target_counts"]["test"].values())),
        ("Quarantined source-related images", audit["quarantine_images"]),
        ("Verified neural final fits", sum(len(item["seeds"]) for item in fits["neural"].values())),
        ("Verified final probes", len(fits["probes"])),
        ("Official test-manifest opens", gate["official_test_manifest_open_count"]),
        ("Test rows used for selection", 0),
    ],
    columns=["Control", "Recorded value"],
)
display(controls)

,Control,Recorded value
0,Clean development images,13285
1,Clean held-out test images,3329
2,Quarantined source-related images,125
3,Verified neural final fits,9
4,Verified final probes,3
5,Official test-manifest opens,1
6,Test rows used for selection,0


## 2. Held-out POLAR result

The primary metric is macro-F1. Confidence intervals use 10,000 class-stratified
bootstrap resamples. The ensemble and all component rows were predeclared; the table is
not a post-test leaderboard.

![Predeclared held-out candidates](assets/polar_test_comparison.png)

In [3]:
names = {
    "locked_ensemble": "Locked ensemble",
    "dinov2_base_multilayer_rbf": "DINOv2-B + RBF SVM",
    "dinov2_base_multilayer_logistic": "DINOv2-B + logistic",
    "dinov2_base_top4": "DINOv2-B top 4",
    "dinov2_small_moderate": "DINOv2-S full",
    "convnext_small_full": "ConvNeXt-S full",
}
metrics = pd.read_csv(ROOT / "results" / "polar_test_metrics.csv")
intervals = load_json("polar_test_uncertainty.json")
metrics["macro_f1_ci"] = [
    f"[{intervals[key]['ci_95_low']:.3f}, {intervals[key]['ci_95_high']:.3f}]"
    for key in metrics["candidate"]
]
metrics["candidate"] = metrics["candidate"].map(names)
display(metrics[["candidate", "macro_f1", "macro_f1_ci", "accuracy", "log_loss", "ece"]])

,candidate,macro_f1,macro_f1_ci,accuracy,log_loss,ece
0,Locked ensemble,0.9399,"[0.931, 0.948]",0.9456,0.1564,0.0291
1,DINOv2-B + RBF SVM,0.9274,"[0.918, 0.936]",0.9342,0.2280,0.0269
2,DINOv2-B + logistic,0.9258,"[0.916, 0.935]",0.9324,0.1764,0.0133
3,DINOv2-B top 4,0.9252,"[0.916, 0.934]",0.9327,0.2173,0.0420
4,DINOv2-S full,0.9131,"[0.903, 0.923]",0.9210,0.2389,0.0298
5,ConvNeXt-S full,0.8914,"[0.880, 0.902]",0.8994,0.3081,0.0429


![Locked ensemble confusion matrix](assets/polar_confusion_matrix.png)

In [4]:
per_class = pd.read_csv(ROOT / "results" / "polar_test_per_class.csv")
per_class = per_class[per_class["candidate"].eq("locked_ensemble")]
display(per_class[["class", "precision", "recall", "f1", "support"]].reset_index(drop=True))

secondary = pd.read_csv(ROOT / "results" / "polar_test_secondary_metrics.csv")
display(secondary[["candidate", "macro_f1", "accuracy", "log_loss", "ece"]])

,class,precision,recall,f1,support
0,running,0.9567,0.9606,0.9586,736
1,sitting,0.9922,0.9855,0.9888,1034
2,standing,0.9251,0.9392,0.9321,921
3,walking,0.8869,0.8730,0.8799,638


,candidate,macro_f1,accuracy,log_loss,ece
0,collapsed_locked_four_class,0.9611,0.9622,0.1082,0.0240
1,direct_three_class_probe,0.9531,0.9543,0.1190,0.0101


## 3. What improved performance

The frozen DINOv2-B learning curve isolates the effect of training-set size. Adaptation
and regularization screens then test whether additional complexity earns its place.
Dropout and image augmentation were retained; MixUp, label smoothing, inverse-frequency
weights, and removing random erasing did not improve the relevant seed-42 baseline.

![Frozen DINOv2-B learning curve](assets/polar_scale_curve.png)

In [5]:
scale = pd.DataFrame(load_json("polar_extension_summary.json")["scale_curve"])
display(scale[["actual_train_size", "macro_f1_mean", "accuracy_mean", "log_loss_mean"]])

,actual_train_size,macro_f1_mean,accuracy_mean,log_loss_mean
0,242,0.8487,0.8590,0.4072
1,500,0.8718,0.8810,0.3389
2,1000,0.8810,0.8915,0.3000
3,3000,0.8980,0.9062,0.2511
4,9958,0.9150,0.9222,0.2168


## 4. Linear versus nonlinear final-stage classifiers

The calibrated RBF SVM is the strongest standalone held-out component, but its gain over
logistic regression is small. The RBF artifact is 870.9 MB and took 60.4 minutes to fit;
the logistic artifact is 0.4 MB, fitted in 13.9 seconds, and has better log loss and ECE.
The SVM is useful as a research probe and ensemble component; logistic regression is the
more practical calibrated endpoint.

In [6]:
probe_rows = metrics[metrics["candidate"].isin(["DINOv2-B + RBF SVM", "DINOv2-B + logistic"])].copy()
probe_rows["fit_seconds"] = [3625.2294, 13.9214]
probe_rows["artifact_mb"] = [870.8566, 0.4136]
display(probe_rows[["candidate", "macro_f1", "accuracy", "log_loss", "ece", "fit_seconds", "artifact_mb"]])

,candidate,macro_f1,accuracy,log_loss,ece,fit_seconds,artifact_mb
1,DINOv2-B + RBF SVM,0.9274,0.9342,0.2280,0.0269,3625.2294,870.8566
2,DINOv2-B + logistic,0.9258,0.9324,0.1764,0.0133,13.9214,0.4136


## 5. Person-level V-COCO study

The original no-retuning audit identified the external gap: the locked three-class
POLAR ensemble reached 0.961 in-domain macro-F1 and
0.667 on
3,761 unambiguous V-COCO images.

The follow-up keeps the official V-COCO memberships, trains on the target training
split, selects on validation, and opens the official test labels once after locking the
final stack. Two aspect-preserving DINOv2-B person views and five geometry features
raise person-level macro-F1 from
0.7071 to
0.8663.

![Official V-COCO test comparison](assets/vcoco_v2_official_test_comparison.png)

![Person-scale gain](assets/vcoco_v2_scale_gain.png)

![Selective prediction](assets/vcoco_v2_selective_prediction.png)

In [7]:
vcoco_metrics = pd.read_csv(ROOT / "results" / "vcoco_v2" / "official_test_metrics.csv")
vcoco_per_class = pd.read_csv(
    ROOT / "results" / "vcoco_v2" / "official_test_per_class.csv"
)
display(
    vcoco_metrics[
        ["method", "macro_f1", "accuracy", "balanced_accuracy", "log_loss", "ece"]
    ]
)
display(vcoco_per_class[["method", "class", "precision", "recall", "f1", "support"]])

,method,macro_f1,accuracy,balanced_accuracy,log_loss,ece
0,scale_conditioned_stacking,0.8663,0.8795,0.8636,0.2902,0.0076
1,historical_v1_dino,0.7071,0.7010,0.7674,1.6525,0.2288


,method,class,precision,recall,f1,support
0,scale_conditioned_stacking,sitting,0.9438,0.9554,0.9496,1882
1,scale_conditioned_stacking,standing,0.8731,0.8852,0.8791,2962
2,scale_conditioned_stacking,walking_running,0.7913,0.7502,0.7702,1233
3,historical_v1_dino,sitting,0.9029,0.9038,0.9033,1882
4,historical_v1_dino,standing,0.8698,0.4828,0.6209,2962
5,historical_v1_dino,walking_running,0.4429,0.9157,0.5970,1233


## 6. Motion identifiability on Okutama-Action

The sealed confirmation compares the locked static model with an 8-frame, 0.5-second
temporal teacher over 1,771 person instances.
The temporal model changes macro-F1 by
**+0.0396** with a 95% paired
recording-cluster interval of
[+0.0202,
+0.0568]. A fixed 50% routing budget
reaches **0.7817 macro-F1**.

![Static, temporal, and routed confirmation](assets/vcoco_v3_confirmation_comparison.png)

![Fixed-budget temporal routing](assets/vcoco_v3_routing_curve.png)

In [8]:
v3_metrics = pd.read_csv(ROOT / "results/vcoco_v3/confirmation_metrics.csv")
selected_v3 = v3_metrics[
    v3_metrics["family"].isin(
        ["source_only_static", "static", "teacher", "hybrid_budget_0.5"]
    )
]
display(
    selected_v3[
        ["family", "macro_f1", "accuracy", "log_loss", "ece"]
    ].reset_index(drop=True)
)

v3_per_class = pd.read_csv(ROOT / "results/vcoco_v3/confirmation_per_class.csv")
display(
    v3_per_class[v3_per_class["family"].isin(["static", "teacher", "hybrid_budget_0.5"])]
    [["family", "class", "precision", "recall", "f1", "support"]]
    .reset_index(drop=True)
)

,family,macro_f1,accuracy,log_loss,ece
0,static,0.7458,0.7301,0.6404,0.0345
1,teacher,0.7854,0.7708,0.5747,0.0292
2,source_only_static,0.5735,0.5731,0.9946,0.1800
3,hybrid_budget_0.5,0.7817,0.7679,0.5822,0.0281


,family,class,precision,recall,f1,support
0,static,sitting,0.8150,0.8661,0.8398,351
1,static,standing,0.7550,0.6046,0.6715,693
2,static,walking_running,0.6762,0.7840,0.7261,727
3,teacher,sitting,0.8801,0.8575,0.8687,351
4,teacher,standing,0.8061,0.6479,0.7184,693
5,teacher,walking_running,0.7053,0.8459,0.7692,727
6,hybrid_budget_0.5,sitting,0.8768,0.8519,0.8642,351
7,hybrid_budget_0.5,standing,0.8195,0.6291,0.7118,693
8,hybrid_budget_0.5,walking_running,0.6960,0.8597,0.7692,727


## 7. CPTR architecture development

The follow-up architecture keeps the static and temporal anchors frozen and evaluates
center-conditioned residuals, camera-compensated trajectories, and confidence-masked
body-region tokens. The center-plus-parts branch changes fixed-validation macro-F1 from
**0.7806** to
**0.7887**. The matched
recording-grouped OOF comparison moves in the other direction:
**0.7165** for the temporal
baseline and **0.7144** for
the candidate. The promotion gate therefore keeps the temporal ensemble as the default.

Motion nulling reduces the candidate's validation macro-F1 by
**0.0441**,
while the grouped analysis identifies occluded windows as the largest observed failure
mode. The fixed-split gain is retained as development evidence, not promoted as a
generalized improvement.

In [9]:
cptr_headline = pd.read_csv(ROOT / "results/okutama_cptr/headline_metrics.csv")
display(
    cptr_headline[
        ["scope", "model", "samples", "recordings", "macro_f1", "accuracy", "log_loss"]
    ]
)

cptr_subgroups = pd.read_csv(ROOT / "results/okutama_cptr/subgroup_metrics.csv")
display(
    cptr_subgroups[
        cptr_subgroups["scope"].eq("crossfit_oof")
        & cptr_subgroups["subgroup"].isin(["window_clear", "window_occluded", "transition"])
    ][["subgroup", "samples", "baseline_macro_f1", "candidate_macro_f1", "macro_f1_delta"]]
)

,scope,model,samples,recordings,macro_f1,accuracy,log_loss
0,development_validation,v3_temporal_baseline,1383,3,0.7806,0.7780,0.5632
1,development_validation,centre_short_parts,1383,3,0.7887,0.7867,0.5576
2,grouped_crossfit_oof,v3_temporal_baseline,4977,11,0.7165,0.7131,0.7539
3,grouped_crossfit_oof,centre_short_parts,4977,11,0.7144,0.7123,0.7503


,subgroup,samples,baseline_macro_f1,candidate_macro_f1,macro_f1_delta
0,transition,398,0.5062,0.5041,-0.0021
2,window_occluded,412,0.6321,0.6013,-0.0308
3,window_clear,4565,0.7249,0.7264,0.0015


## 8. Attribution: localization is not enough

The fixed 256-image audit combines deletion/insertion, person-box localization,
equal-area person/context occlusion, target sensitivity, and parameter randomization.
ConvNeXt Grad-CAM has stronger causal and sanity evidence. DINOv2-B integrated gradients
localize on people but remain highly correlated after changing the target and resetting
learned layers, so they are not promoted as faithful causal explanations.

![BBox-aware attribution audit](assets/polar_faithfulness.png)

![Target and parameter randomization](assets/polar_attribution_sanity.png)

In [10]:
faith = load_json("polar_faithfulness_summary.json")["aggregate"]
rows = []
for family in ("convnext_small_full", "dinov2_base_top4"):
    rows.append(
        {
            "family": names[family],
            "deletion_selectivity_gap": faith[family]["deletion_selectivity_gap"]["mean"],
            "person_area_lift": faith[family]["person_attribution_mass_lift"]["mean"],
            "person_minus_context_drop": faith[family]["person_minus_context_occlusion_drop"]["mean"],
            "alternative_target_rho": faith[family]["target_vs_alternative_attribution_spearman"]["mean"],
            "randomized_cascade_rho": faith[family]["randomized_adapted_cascade_spearman"]["mean"],
        }
    )
display(pd.DataFrame(rows))

,family,deletion_selectivity_gap,person_area_lift,person_minus_context_drop,alternative_target_rho,randomized_cascade_rho
0,ConvNeXt-S full,0.1634,2.3679,0.2336,-0.4913,0.1347
1,DINOv2-B top 4,0.0582,1.0950,0.0124,0.9341,0.7081


## 9. Bounded bit-flip robustness

Fault injection is reported separately from attribution faithfulness. Exact bit flips
are applied either to the uint8 input tensor or to an int8-quantized classifier weight
matrix. The result measures local prediction stability on the declared cohort; it is not
hardware certification.

![Fault robustness](assets/polar_fault_robustness.png)

In [11]:
fault = pd.DataFrame(load_json("polar_fault_summary.json")["aggregate_results"])
fault = fault[fault["fault_seed"].astype(str).isin(["none", "aggregate"])]
display(
    fault[[
        "family", "condition", "level", "macro_f1",
        "prediction_agreement_with_clean", "mean_absolute_probability_drift"
    ]].reset_index(drop=True)
)

,family,condition,level,macro_f1,prediction_agreement_with_clean,mean_absolute_probability_drift
0,convnext_small_full,clean_float,0.0000,0.8988,1.0000,0.0000
1,convnext_small_full,uint8_input_bit_flip_rate,0.0000,0.8988,1.0000,0.0000
2,convnext_small_full,uint8_input_bit_flip_rate,0.0000,0.9027,0.9961,0.0011
3,convnext_small_full,uint8_input_bit_flip_rate,0.0001,0.9027,0.9961,0.0039
4,convnext_small_full,uint8_input_bit_flip_rate,0.0010,0.8990,0.9844,0.0121
5,convnext_small_full,symmetric_int8_head_weight_bit_flips,0.0000,0.8988,1.0000,0.0001
6,convnext_small_full,symmetric_int8_head_weight_bit_flips,1.0000,0.8988,1.0000,0.0001
7,convnext_small_full,symmetric_int8_head_weight_bit_flips,4.0000,0.8988,1.0000,0.0003
8,convnext_small_full,symmetric_int8_head_weight_bit_flips,16.0000,0.8988,1.0000,0.0009
9,dinov2_base_top4,clean_float,0.0000,0.9533,1.0000,0.0000


## 10. Conclusions

- Data scale is the largest isolated performance amplifier in this study.
- DINOv2-B representations support strong linear and nonlinear final-stage classifiers.
- Development-locked model diversity produces a statistically supported ensemble gain.
- Person-centric scale conditioning raises official-test V-COCO macro-F1 from 0.7071
  to 0.8663 and greatly reduces the observed association with apparent person size.
- Factorized posture-motion targets add a smaller, independently supported gain under
  matched feature inputs.
- Short temporal context improves sealed Okutama macro-F1 from 0.7458 to 0.7854; fixed
  50% routing retains 0.7817.
- The center-plus-parts residual improves one fixed split but not recording-grouped OOF,
  showing why the grouped promotion gate is necessary.
- ConvNeXt Grad-CAM passes the declared sanity checks more convincingly than DINOv2-B
  integrated gradients.

The complete method, evidence lineage, discussion, and references are documented in
`docs/POLAR_PUBLIC_REPORT.md`, `docs/VCOCO_V2_EXTERNAL_TRANSFER.md`,
`docs/VCOCO_V3_MOTION_IDENTIFIABILITY.md`, and `docs/OKUTAMA_CPTR_DEVELOPMENT.md`.